# Module 5 Capstone Component 5.1
## Analysing How Data Splitting Affects Model Performance

**Author:** Avdhesh Thakur  
**Course:** Professional Certificate in Machine Learning and AI – Imperial College London  

---

### Overview

In this notebook, I experiment with two different data splitting strategies on the Wine dataset:

- **Experiment 1:** 60% training, 20% validation, 20% test
- **Experiment 2:** 70% training, 15% validation, 15% test

The goal is to understand how the choice of split ratio affects the model's validation and test accuracy,
and what that means for building reliable ML models.

The model used in both experiments is **Logistic Regression** from scikit-learn.

In [ ]:
##############################################################
# Import the libraries needed for this notebook
#
# - sklearn.datasets   : to load the built-in Wine dataset
# - train_test_split   : to split data into train/val/test sets
# - LogisticRegression : the classification model used
# - accuracy_score     : to measure overall prediction accuracy
# - classification_report : for a detailed per-class breakdown
##############################################################

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
##############################################################
# Load the Wine dataset
#
# The Wine dataset is a built-in scikit-learn dataset.
# It contains 178 samples of wine, each described by
# 13 chemical features (e.g. alcohol content, magnesium level).
# The target variable has 3 classes representing
# three different cultivars of wine.
#
# X = feature matrix (178 rows x 13 columns)
# y = target labels (0, 1, or 2)
##############################################################

data = load_wine()

X = data.data
y = data.target

print(f"Total samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Classes: {list(data.target_names)}")

---
## Experiment 1: 60% Training | 20% Validation | 20% Test

This split gives more data to the validation and test sets,
which produces a more reliable performance estimate on unseen data.
The trade-off is that the model trains on fewer samples.

In [ ]:
##############################################################
# Split the data into 60% train, 20% validation, 20% test
#
# Since train_test_split() can only split into two groups
# at a time, we do this in two steps:
#
# Step 1: Set aside 20% for the final test set.
#         The remaining 80% goes into train+validation.
#
# Step 2: Split the 80% further into 75% train and 25% val.
#         25% of 80% = 20% of the full dataset.
#         This gives us the final 60/20/20 split.
#
# stratify=y ensures each class is proportionally
# represented in every split.
##############################################################

# Step 1: Hold out 20% for testing
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Step 2: Split remaining 80% into 75% train and 25% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    random_state=42,
    stratify=y_train_val
)

print(f"Training samples   : {len(X_train)}")
print(f"Validation samples : {len(X_val)}")
print(f"Test samples       : {len(X_test)}")

In [ ]:
##############################################################
# Train the Logistic Regression model
#
# max_iter=5000 is set to ensure the model has enough
# iterations to converge properly on this dataset.
# random_state=42 makes the results reproducible.
#
# The model is trained ONLY on the training set.
# The validation and test sets are not seen at this stage.
##############################################################

model_exp1 = LogisticRegression(
    max_iter=5000,
    random_state=42
)

model_exp1.fit(X_train, y_train)

In [ ]:
##############################################################
# Evaluate the model on the validation set
#
# The validation set is used to check how well the model
# performs on data it has not been trained on.
# This helps us decide whether the model is good enough
# before we touch the test set.
##############################################################

val_preds_exp1 = model_exp1.predict(X_val)

val_accuracy_exp1 = accuracy_score(y_val, val_preds_exp1)

print(f"Experiment 1 - Validation Accuracy: {val_accuracy_exp1:.4f}")

In [ ]:
##############################################################
# Evaluate the model on the test set
#
# The test set is only used once, at the very end.
# It gives an unbiased estimate of how the model
# will perform on completely new, real-world data.
##############################################################

test_preds_exp1 = model_exp1.predict(X_test)

test_accuracy_exp1 = accuracy_score(y_test, test_preds_exp1)

print(f"Experiment 1 - Test Accuracy: {test_accuracy_exp1:.4f}")

In [ ]:
##############################################################
# Print the detailed classification report for Experiment 1
#
# This shows per-class metrics:
# - Precision : of all predicted positives, how many were correct
# - Recall    : of all actual positives, how many were found
# - F1-score  : harmonic mean of precision and recall
# - Support   : number of actual samples per class in the test set
##############################################################

print("\nClassification Report on Test Set (Experiment 1 - 60/20/20):")
print(
    classification_report(
        y_test,
        test_preds_exp1,
        target_names=data.target_names
    )
)

---
## Experiment 2: 70% Training | 15% Validation | 15% Test

This split gives the model more training data.
The validation and test sets are smaller, which can sometimes
lead to overoptimistic accuracy scores due to less representation.

In [ ]:
##############################################################
# Split the data into 70% train, 15% validation, 15% test
#
# Step 1: Set aside 15% for the final test set.
#         The remaining 85% goes into train+validation.
#
# Step 2: From the remaining 85%, take 17.65% for validation.
#         17.65% of 85% ≈ 15% of the full dataset.
#         This gives the final 70/15/15 split.
##############################################################

# Step 1: Hold out 15% for testing
X_train_val2, X_test2, y_train_val2, y_test2 = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# Step 2: Split remaining 85% into ~82.35% train and ~17.65% validation
X_train2, X_val2, y_train2, y_val2 = train_test_split(
    X_train_val2,
    y_train_val2,
    test_size=0.1765,
    random_state=42,
    stratify=y_train_val2
)

print(f"Training samples   : {len(X_train2)}")
print(f"Validation samples : {len(X_val2)}")
print(f"Test samples       : {len(X_test2)}")

In [ ]:
##############################################################
# Train the Logistic Regression model for Experiment 2
#
# Same model configuration as Experiment 1 so that
# the only variable changing between experiments
# is the data split ratio.
##############################################################

model_exp2 = LogisticRegression(
    max_iter=5000,
    random_state=42
)

model_exp2.fit(X_train2, y_train2)

In [ ]:
##############################################################
# Evaluate the model on the validation set (Experiment 2)
##############################################################

val_preds_exp2 = model_exp2.predict(X_val2)

val_accuracy_exp2 = accuracy_score(y_val2, val_preds_exp2)

print(f"Experiment 2 - Validation Accuracy: {val_accuracy_exp2:.4f}")

In [ ]:
##############################################################
# Evaluate the model on the test set (Experiment 2)
##############################################################

test_preds_exp2 = model_exp2.predict(X_test2)

test_accuracy_exp2 = accuracy_score(y_test2, test_preds_exp2)

print(f"Experiment 2 - Test Accuracy: {test_accuracy_exp2:.4f}")

In [ ]:
##############################################################
# Print the detailed classification report for Experiment 2
##############################################################

print("\nClassification Report on Test Set (Experiment 2 - 70/15/15):")
print(
    classification_report(
        y_test2,
        test_preds_exp2,
        target_names=data.target_names
    )
)

---
## Summary of Results

| Split        | Train Samples | Val Samples | Test Samples | Val Accuracy | Test Accuracy |
|-------------|--------------|------------|-------------|-------------|---------------|
| 60/20/20    | 106          | 36         | 36          | 94.44%      | 97.22%        |
| 70/15/15    | 124          | 27         | 27          | 100.00%     | 96.30%        |

---

## Key Observations

**60/20/20 split:**  
The validation accuracy was 94.44% and the test accuracy was 97.22%. The larger validation and test sets give a more reliable picture of how the model will perform on unseen data. The gap between validation and test accuracy is small, which suggests consistent performance.

**70/15/15 split:**  
The validation accuracy was 100%, which looks perfect but is misleading. The validation set only had 27 samples, which is too small to represent the full data distribution reliably. The test accuracy of 96.30% is still strong, but the 100% validation score is overoptimistic.

**What happens without a validation set?**  
Without a validation set, the only way to compare models is to evaluate them on the test set directly. This means the test set gets used for decision-making, which removes its role as a truly unseen benchmark. The final performance estimate would no longer be trustworthy.

**Application to capstone project:**  
For my NHS surgical scheduling capstone project, I would use a 60/20/20 split to ensure the validation and test sets are large enough to give reliable results. I would also split the data by time rather than randomly, so the model is always evaluated on future data, which better reflects real deployment conditions.